In [ ]:
# Databricks notebook sourceMAGIC %sqlselect * from bupa_bronze.claims

%md# Goal: Claims are the messiest—fix dates, impute critical nulls, enforce FK to Policies/Providers, quarantine orphans, dedupe, and partition by year for performance.## Steps### 1.Read Bronze### 2.Type casting & selection### 3.Key check (Claim_ID) → quarantine### 4.Fix dates, impute payout for Settled where missing### 5.FK checks: Policy_ID and Provider_ID (provider may be null)### 6.Deduplicate (Claim_ID by most recent Date_Reported then Date_Settled)### 7.Partitioned write by Year_Reported + register

In [ ]:
MAGIC %run /Workspace/Repos/karthikvanapalli1505@gmail.com/Bupa_Insuarnce_Project/02_Silver_Layer/_00_utils_silver

In [ ]:
# ============ CLAIMS → SILVER ============clm_bz = spark.read.format("delta").load(paths_bronze["claims"])clm = (clm_bz.select(    F.col("Claim_ID").cast("string"),    F.col("Policy_ID").cast("string"),    F.col("Provider_ID").cast("string"),    F.to_date("Date_Reported").alias("Date_Reported"),    F.to_date("Date_Settled").alias("Date_Settled"),    F.col("Claim_Amount_GBP").cast("double"),    F.col("Payout_GBP").cast("double"),    F.col("Fraud_Label").cast("int"),    F.col("Claim_Type").cast("string"),    F.col("Claim_Status").cast("string")))# Key check → quarantine null claim_idclm_bad = clm.filter(F.col("Claim_ID").isNull())clm_ok  = clm.filter(F.col("Claim_ID").isNotNull())if clm_bad.take(1): quarantine(clm_bad, "null_claim_id", "claims")# Date fixesclm_ok = fix_dates(clm_ok, "Date_Reported", "Date_Settled")# Impute payout for Settled claims if null → 0.0 (business rule)clm_ok = clm_ok.withColumn(    "Payout_GBP",    F.when(F.col("Payout_GBP").isNull() & (F.col("Claim_Status")=="Settled"), F.lit(0.0))     .otherwise(F.col("Payout_GBP")))# Enforce FK to policies (mandatory)pol_keys = spark.table(f"{CATALOG_SILVER}.policies").select("Policy_ID").dropDuplicates()clm_ok = fk_filter(clm_ok, "Policy_ID", pol_keys, "Policy_ID", "claims", "fk_policy_missing")# Enforce FK to providers (optional when Provider_ID is present)prov_keys = spark.table(f"{CATALOG_SILVER}.providers").select("Provider_ID").dropDuplicates()# Keep rows where Provider_ID is null (allowed) OR valid reference; quarantine only those with non-null Provider_ID not foundjoined = clm_ok.join(prov_keys.withColumnRenamed("Provider_ID","_Provider_ID"),                     (clm_ok["Provider_ID"] == F.col("_Provider_ID")),                     "left")bad_prov = joined.filter(F.col("Provider_ID").isNotNull() & F.col("_Provider_ID").isNull())good_prov = joined.filter(F.col("_Provider_ID").isNotNull() | F.col("Provider_ID").isNull()).drop("_Provider_ID")if bad_prov.take(1):    quarantine(bad_prov.drop("_Provider_ID"), "fk_provider_missing", "claims")# Dedupe on Claim_ID by most recent Date_Reported then Date_Settledclm_dedup = drop_dupes_keep_latest(good_prov, ["Claim_ID"], ["Date_Reported","Date_Settled"])# Partition by Year_Reportedclm_silver = clm_dedup.withColumn("Year_Reported", F.year("Date_Reported"))(clm_silver.write.format("delta").mode("overwrite").partitionBy("Year_Reported")  .save(paths_silver["claims"]))#spark.sql(f"CREATE TABLE IF NOT EXISTS {CATALOG_SILVER}.claims USING DELTA LOCATION '{paths_silver['claims']}'")clm_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("bupa_silver.claims")print("Claims → Silver complete.")

In [ ]:
MAGIC %sqlSELECT * FROM bupa_silver.claims

In [ ]:
from pyspark.sql import functions as F# Load tablesbronze = spark.table("bupa_bronze.claims")silver = spark.table("bupa_silver.claims")# List of columns to compare (excluding partition columns if needed)compare_cols = [    "Claim_ID", "Policy_ID", "Provider_ID", "Date_Reported", "Date_Settled",    "Claim_Amount_GBP", "Payout_GBP", "Fraud_Label", "Claim_Type", "Claim_Status"]# Alias columns for claritybronze_alias = [F.col(c).alias(f"bronze_{c}") for c in compare_cols]silver_alias = [F.col(c).alias(f"silver_{c}") for c in compare_cols]# Select and join on Claim_IDbronze_df = bronze.select(*bronze_alias)silver_df = silver.select(*silver_alias)joined = bronze_df.join(    silver_df,    bronze_df.bronze_Claim_ID == silver_df.silver_Claim_ID,    "full_outer")# For each feature, show if it changeddiff_exprs = []for c in compare_cols:    if c == "Claim_ID":        continue    diff_exprs.append(        F.when(            F.col(f"bronze_{c}").isNull() & F.col(f"silver_{c}").isNotNull(), "added"        ).when(            F.col(f"bronze_{c}").isNotNull() & F.col(f"silver_{c}").isNull(), "removed"        ).when(            F.col(f"bronze_{c}") != F.col(f"silver_{c}"), "changed"        ).otherwise("same").alias(f"{c}_diff")    )result = joined.select(    F.coalesce(F.col("bronze_Claim_ID"), F.col("silver_Claim_ID")).alias("Claim_ID"),    *[F.col(f"bronze_{c}") for c in compare_cols if c != "Claim_ID"],    *[F.col(f"silver_{c}") for c in compare_cols if c != "Claim_ID"],    *diff_exprs)display(result)

In [ ]:
from pyspark.sql import functions as F# List of features to summarize (exclude Claim_ID)features = [    "Policy_ID", "Provider_ID", "Date_Reported", "Date_Settled",    "Claim_Amount_GBP", "Payout_GBP", "Fraud_Label", "Claim_Type", "Claim_Status"]# For each feature, count the diff typessummary_exprs = []for feat in features:    for diff_type in ["changed", "added", "removed"]:        summary_exprs.append(            F.sum(F.when(F.col(f"{feat}_diff") == diff_type, 1).otherwise(0)).alias(f"{feat}_{diff_type}")        )summary = result.agg(*summary_exprs)display(summary)